In [39]:
import warnings
warnings.filterwarnings('ignore')

In [40]:
import pandas as pd, numpy as np
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns


In [41]:
url="https://raw.githubusercontent.com/nikitaguptasrivastava-cell/Information-Assurance/refs/heads/main/Baseline_Model_Dataset_csv.csv"
df=pd.read_csv(url)
df.head()

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Time_of_Transaction,Device_Used,Location,Previous_Fraudulent_Transactions,Account_Age,Number_of_Transactions_Last_24H,Payment_Method,Fraudulent
0,T1222,1519,2000.00,Online Purchase,10,Desktop,Miami,3,90,1,Credit Card,0
1,T1473,4471,3090.95,Online Purchase,21,Mobile,Seattle,0,76,14,Credit Card,0
2,T1782,1194,2647.53,Bank Transfer,6,Mobile,Chicago,4,53,12,Credit Card,0
3,T2875,2054,2616.57,Online Purchase,6,Tablet,Los Angeles,0,115,14,Credit Card,0
4,T4541,4954,4657.24,ATM Withdrawal,22,Desktop,New York,2,52,10,Credit Card,0


In [42]:
df.shape


(9, 12)

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 12 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Transaction_ID                    9 non-null      object 
 1   User_ID                           9 non-null      int64  
 2   Transaction_Amount                9 non-null      float64
 3   Transaction_Type                  9 non-null      object 
 4   Time_of_Transaction               9 non-null      int64  
 5   Device_Used                       9 non-null      object 
 6   Location                          9 non-null      object 
 7   Previous_Fraudulent_Transactions  9 non-null      int64  
 8   Account_Age                       9 non-null      int64  
 9   Number_of_Transactions_Last_24H   9 non-null      int64  
 10  Payment_Method                    9 non-null      object 
 11  Fraudulent                        9 non-null      int64  
dtypes: float64(1

In [44]:
df.describe()

,User_ID,Transaction_Amount,Time_of_Transaction,Previous_Fraudulent_Transactions,Account_Age,Number_of_Transactions_Last_24H,Fraudulent
count,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000,9.000000
mean,3006.777778,2322.775556,12.111111,2.111111,70.222222,10.111111,0.111111
std,1646.221642,1398.360737,7.623064,1.536591,28.547232,4.166667,0.333333
min,1002.000000,14.610000,0.000000,0.000000,43.000000,1.000000,0.000000
25%,1519.000000,2000.000000,6.000000,1.000000,49.000000,10.000000,0.000000
50%,2638.000000,2616.570000,12.000000,2.000000,53.000000,11.000000,0.000000
75%,4535.000000,3081.490000,20.000000,3.000000,90.000000,12.000000,0.000000
max,4954.000000,4657.240000,22.000000,4.000000,115.000000,14.000000,1.000000


In [45]:
# checking the columns for null values
df.isnull().sum()

,0
Transaction_ID,0
User_ID,0
Transaction_Amount,0
Transaction_Type,0
Time_of_Transaction,0
Device_Used,0
Location,0
Previous_Fraudulent_Transactions,0
Account_Age,0
Number_of_Transactions_Last_24H,0


In [46]:
#Developing synthetic dataset
#Finding fraudscore
#Normalizing dataset
#Normalizing the columns by creating a normalize function
def normalize(columns):
  return (columns - columns.min()) / (columns.max() - columns.min())

df["Normalized_Amt"] = normalize(df["Transaction_Amount"])

df["Normalized_History"] = normalize(df["Previous_Fraudulent_Transactions"])

df["Normalized_Frequency"] = normalize(df["Number_of_Transactions_Last_24H"])

df["Normalized_Time"] = normalize(df["Time_of_Transaction"])

df["Nomalized_Age"] = 1 - normalize(df["Account_Age"])

#credit_fraud_detection_dataset.head()
#using the probability of occurence and for example fraud_rate/max_fraud_rate so risk score varies between 0 to 1,calculated the transaction type risk
Transaction_Type_Risk = {
    "Bill Payment":0.25,
    "ATM Withdrawal":0.25,
    "POS Payment":0.25,
    "Bank Transfer":0.5,
    "Online Purchase":1
}
df["Normalized_Transaction_Type"] = df["Transaction_Type"].map(Transaction_Type_Risk)
device_risk = {
    "Mobile":1,
    "Desktop":0.594,
    "Tablet":0.2
}
df["Normalized_Device"]= df["Device_Used"].map(device_risk)

Location_risk = {
    "Boston":1,
    "Seattle":0.5,
    "Chicago":0.5,
    "Houston":0.5,
    "New York":0.5,
    "Los Angeles":0.5,
    "Miami":0.5,
    "San Francisco":0.5
}

df["Normalized_Location"]= df["Location"].map(Location_risk)
df.head()


,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Time_of_Transaction,Device_Used,Location,Previous_Fraudulent_Transactions,Account_Age,Number_of_Transactions_Last_24H,Payment_Method,Fraudulent,Normalized_Amt,Normalized_History,Normalized_Frequency,Normalized_Time,Nomalized_Age,Normalized_Transaction_Type,Normalized_Device,Normalized_Location
0,T1222,1519,2000.00,Online Purchase,10,Desktop,Miami,3,90,1,Credit Card,0,0.427643,0.75,0.000000,0.454545,0.347222,1.00,0.594,0.5
1,T1473,4471,3090.95,Online Purchase,21,Mobile,Seattle,0,76,14,Credit Card,0,0.662629,0.00,1.000000,0.954545,0.541667,1.00,1.000,0.5
2,T1782,1194,2647.53,Bank Transfer,6,Mobile,Chicago,4,53,12,Credit Card,0,0.567118,1.00,0.846154,0.272727,0.861111,0.50,1.000,0.5
3,T2875,2054,2616.57,Online Purchase,6,Tablet,Los Angeles,0,115,14,Credit Card,0,0.560450,0.00,1.000000,0.272727,0.000000,1.00,0.200,0.5
4,T4541,4954,4657.24,ATM Withdrawal,22,Desktop,New York,2,52,10,Credit Card,0,1.000000,0.50,0.692308,1.000000,0.875000,0.25,0.594,0.5


In [47]:
df.isnull().sum()

,0
Transaction_ID,0
User_ID,0
Transaction_Amount,0
Transaction_Type,0
Time_of_Transaction,0
Device_Used,0
Location,0
Previous_Fraudulent_Transactions,0
Account_Age,0
Number_of_Transactions_Last_24H,0


In [48]:
df=df.drop(columns=['Transaction_ID','User_ID','Transaction_Amount','Transaction_Type','Time_of_Transaction','Device_Used','Location','Previous_Fraudulent_Transactions',
                    'Account_Age','Number_of_Transactions_Last_24H','Payment_Method'])

In [49]:
df.head()

,Fraudulent,Normalized_Amt,Normalized_History,Normalized_Frequency,Normalized_Time,Nomalized_Age,Normalized_Transaction_Type,Normalized_Device,Normalized_Location
0,0,0.427643,0.75,0.000000,0.454545,0.347222,1.00,0.594,0.5
1,0,0.662629,0.00,1.000000,0.954545,0.541667,1.00,1.000,0.5
2,0,0.567118,1.00,0.846154,0.272727,0.861111,0.50,1.000,0.5
3,0,0.560450,0.00,1.000000,0.272727,0.000000,1.00,0.200,0.5
4,0,1.000000,0.50,0.692308,1.000000,0.875000,0.25,0.594,0.5


In [50]:
y=df['Fraudulent']
y.head()

,Fraudulent
0,0
1,0
2,0
3,0
4,0


In [51]:
df_without_fraudulent=df.drop(columns=['Fraudulent'])
df_without_fraudulent.head()

,Normalized_Amt,Normalized_History,Normalized_Frequency,Normalized_Time,Nomalized_Age,Normalized_Transaction_Type,Normalized_Device,Normalized_Location
0,0.427643,0.75,0.000000,0.454545,0.347222,1.00,0.594,0.5
1,0.662629,0.00,1.000000,0.954545,0.541667,1.00,1.000,0.5
2,0.567118,1.00,0.846154,0.272727,0.861111,0.50,1.000,0.5
3,0.560450,0.00,1.000000,0.272727,0.000000,1.00,0.200,0.5
4,1.000000,0.50,0.692308,1.000000,0.875000,0.25,0.594,0.5


In [53]:
model = LinearRegression()
model.fit(df_without_fraudulent,y)
print(model.score(df_without_fraudulent,y))
print(model.coef_)
print(f"Error: {model.intercept_:.4f}")

1.0
[ 0.69889365 -0.63714044 -0.42977976 -0.79748811  0.66212057  0.23496092
  0.14210112  1.58230292]
Error: -0.7990
